In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp  

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional

In [14]:
"""
EV Smart Charging — Hindsight-Optimal Schedule Calculator
=========================================================
Calculates the cheapest possible charging schedules for a fleet of EVs
given historical EPEX SPOT day-ahead electricity prices.

Data inputs:
  1. EV telematics CSV (Enode/smartcar format, 59 columns)
  2. EPEX SPOT auction prices CSV (Switzerland day-ahead, hourly)

Usage:
  python smart_charge_optimizer.py \
      --ev-data smart_charge_ev_2026.csv \
      --spot-prices auction_spot_prices_switzerland_2026.csv \
      --output results.csv

"""

from __future__ import annotations

import argparse
import csv
import io as _io
import math
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional

import pandas as pd
import numpy as np


# ═══════════════════════════════════════════════════════════════════════════════
# DOMAIN MODELS
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class Vehicle:
    """Static attributes for one EV in the fleet."""
    vehicle_id: str
    brand: str
    model: str
    year: Optional[int] = None
    battery_capacity_kwh: Optional[float] = None

    @property
    def display_name(self) -> str:
        return f"{self.brand} {self.model}"


@dataclass
class ChargingSession:
    """
    One contiguous plug-in → plug-out window for a single vehicle.

    Constructed by SessionExtractor from sequential state-change events.
    """
    vehicle_id: str
    plug_in_time: datetime
    plug_out_time: datetime
    soc_start_pct: float          # battery % when session started
    soc_target_pct: float         # chargelimit (target SoC %)
    charge_rate_kw: float         # estimated_charging_power
    battery_capacity_kwh: float   # vehicle_chargestate_batterycapacity

    @property
    def energy_needed_kwh(self) -> float:
        """kWh that must be delivered in this session."""
        delta_pct = self.soc_target_pct - self.soc_start_pct
        if delta_pct <= 0:
            return 0.0
        return delta_pct / 100.0 * self.battery_capacity_kwh

    @property
    def hours_needed(self) -> float:
        """Minimum hours of charging required at the given rate."""
        if self.charge_rate_kw <= 0:
            return 0.0
        return self.energy_needed_kwh / self.charge_rate_kw

    @property
    def hours_available(self) -> float:
        """Total hours the car is plugged in."""
        return (self.plug_out_time - self.plug_in_time).total_seconds() / 3600.0

    @property
    def flexibility_ratio(self) -> float:
        """Ratio of available time to required charging time. >1 means slack."""
        if self.hours_needed <= 0:
            return float("inf")
        return self.hours_available / self.hours_needed


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 — DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

class EVDataLoader:
    """
    Parses the raw EV telematics CSV.

    The CSV has 59 columns. Two of them (vehicle_scopes, updatedfields)
    contain quoted strings with internal commas, e.g.:
        "{vehicle:control:charging,vehicle:read:data}"
    A naive str.split(",") would break these. We use Python's csv module
    (via pandas) which handles RFC-4180 quoting correctly.

    Additionally, the exported file wraps each row in an extra outer quote layer.
    We strip that first before handing the data to pandas.
    """

    RELEVANT_COLS = [
        "createdat",
        "vehicle_id",
        "vehicle_information_brand",
        "vehicle_information_model",
        "vehicle_information_year",
        "vehicle_chargestate_batterylevel",
        "vehicle_chargestate_batterycapacity",
        "vehicle_chargestate_chargelimit",
        "vehicle_chargestate_powerdeliverystate",
        "vehicle_chargestate_chargerate",
        "vehicle_chargestate_chargetimeremaining",
        "estimated_charging_power",
        "plugin_time",
        "last_departure_time",
        "last_plugin_time",
    ]

    def load(self, filepath: str | Path) -> pd.DataFrame:
        """Read the EV CSV, return a typed DataFrame sorted by (vehicle_id, time)."""
        print(f"[EVDataLoader] Reading {filepath}...")

        # The file wraps each row in an extra outer quote layer (non-standard export).
        # csv.reader strips the outer quotes and unescapes inner "" → " per RFC-4180,
        # giving us a clean CSV string that pandas can parse normally.
        with open(filepath, "r", encoding="utf-8") as f:
            raw_rows = [row[0] for row in csv.reader(f) if row]
        clean_csv = _io.StringIO("\n".join(raw_rows))

        df = pd.read_csv(clean_csv, usecols=self.RELEVANT_COLS, low_memory=False)
        print(f"[EVDataLoader] Loaded {len(df):,} rows, {df['vehicle_id'].nunique()} vehicles.")

        # ── Type conversions ──
        df["createdat"] = pd.to_datetime(df["createdat"], utc=True)

        for col in ["plugin_time", "last_departure_time", "last_plugin_time"]:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

        for col in [
            "vehicle_chargestate_batterylevel",
            "vehicle_chargestate_batterycapacity",
            "vehicle_chargestate_chargelimit",
            "vehicle_chargestate_chargerate",
            "vehicle_chargestate_chargetimeremaining",
            "estimated_charging_power",
        ]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # ── Sort chronologically per vehicle ──
        df.sort_values(["vehicle_id", "createdat"], inplace=True)
        df.reset_index(drop=True, inplace=True)

        return df


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — SESSION EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

class SessionExtractor:
    """
    Walks each vehicle's event stream and detects plug-in → plug-out cycles.

    State machine:
        UNPLUGGED  →  PLUGGED_IN:CHARGING  →  PLUGGED_IN:COMPLETE  →  UNPLUGGED
                            ↑                         |
                            └── PLUGGED_IN:STOPPED ──┘
    """

    PLUGGED_STATES = frozenset({
        "PLUGGED_IN:CHARGING",
        "PLUGGED_IN:COMPLETE",
        "PLUGGED_IN:STOPPED",
    })

    MIN_SESSION_HOURS = 0.5       # ignore sessions shorter than 30 min
    MIN_ENERGY_KWH = 0.1          # ignore sessions with negligible energy need

    def __init__(self, vehicles: dict[str, Vehicle] | None = None):
        self.vehicles = vehicles or {}

    def extract(self, events: pd.DataFrame) -> list[ChargingSession]:
        """Return a list of ChargingSession objects from the event stream."""
        sessions: list[ChargingSession] = []

        for vehicle_id, group in events.groupby("vehicle_id"):
            vehicle_sessions = self._extract_vehicle(vehicle_id, group)
            sessions.extend(vehicle_sessions)

        print(f"[SessionExtractor] Extracted {len(sessions):,} charging sessions.")
        return sessions

    def _extract_vehicle(
        self, vehicle_id: str, group: pd.DataFrame
    ) -> list[ChargingSession]:
        """Walk one vehicle's events and emit sessions."""
        sessions: list[ChargingSession] = []
        session_start_row = None

        for _, row in group.iterrows():
            state = row["vehicle_chargestate_powerdeliverystate"]

            if state in self.PLUGGED_STATES and session_start_row is None:
                session_start_row = row

            elif state == "UNPLUGGED" and session_start_row is not None:
                session = self._build_session(vehicle_id, session_start_row, row)
                if session is not None:
                    sessions.append(session)
                session_start_row = None

        return sessions

    def _build_session(
        self,
        vehicle_id: str,
        start_row: pd.Series,
        end_row: pd.Series,
    ) -> ChargingSession | None:
        """Construct a ChargingSession from start/end event rows, or None if invalid."""
        plug_in = start_row["createdat"]
        plug_out = end_row["createdat"]
        duration_h = (plug_out - plug_in).total_seconds() / 3600.0

        if duration_h < self.MIN_SESSION_HOURS:
            return None

        soc_start = start_row["vehicle_chargestate_batterylevel"]
        soc_target = start_row["vehicle_chargestate_chargelimit"]
        capacity = start_row["vehicle_chargestate_batterycapacity"]
        rate = start_row["estimated_charging_power"]

        # Validate numerics
        if pd.isna(soc_start) or pd.isna(soc_target) or pd.isna(capacity) or pd.isna(rate):
            return None
        if rate <= 0 or capacity <= 0:
            return None

        energy = (soc_target - soc_start) / 100.0 * capacity
        if energy < self.MIN_ENERGY_KWH:
            return None

        return ChargingSession(
            vehicle_id=vehicle_id,
            plug_in_time=plug_in,
            plug_out_time=plug_out,
            soc_start_pct=soc_start,
            soc_target_pct=soc_target,
            charge_rate_kw=rate,
            battery_capacity_kwh=capacity,
        )


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SPOT PRICE PROVIDER
# ═══════════════════════════════════════════════════════════════════════════════

class SpotPriceProvider:
    """
    Loads EPEX SPOT day-ahead auction prices and provides O(1) hourly lookup.

    Expected CSV format (first row is a comment, second row is the header):
        # 09/03/2026 10:10:03 AM: Prices - EPEX Spot Market Auction ...
        Delivery day,Hour 1,Hour 2,Hour 3A,Hour 3B,Hour 4,...,Hour 24,...

    Hour 3A/3B handle DST transitions (Hour 3B is only populated on the
    "spring forward" day when there's a 25th hour).
    """

    def __init__(self, filepath: str | Path):
        self.prices: dict[str, float] = {}  # "YYYY-MM-DDTHH" -> EUR/MWh
        self._load(filepath)

    def _load(self, filepath: str | Path) -> None:
        """Parse the EPEX CSV into an hourly price dictionary."""
        df = pd.read_csv(filepath, skiprows=1)

        # Map column names to hour offsets (0-indexed from midnight)
        hour_map: dict[str, int] = {"Hour 3A": 2}
        for h in range(1, 25):
            col = f"Hour {h}"
            if h <= 2:
                hour_map[col] = h - 1      # Hour 1 → 00:00, Hour 2 → 01:00
            elif h == 3:
                continue                     # Handled by Hour 3A
            else:
                hour_map[col] = h - 1       # Hour 4 → 03:00, ..., Hour 24 → 23:00

        for _, row in df.iterrows():
            day = pd.to_datetime(row["Delivery day"], format="%d/%m/%Y")
            for col, hour_offset in hour_map.items():
                if col not in df.columns:
                    continue
                val = row.get(col)
                if pd.notna(val):
                    ts = day + timedelta(hours=hour_offset)
                    key = ts.strftime("%Y-%m-%dT%H")
                    self.prices[key] = float(val)

            # Handle Hour 3B (DST, 25th hour — rare)
            if "Hour 3B" in df.columns and pd.notna(row.get("Hour 3B")):
                ts = day + timedelta(hours=2, minutes=30)
                key = ts.strftime("%Y-%m-%dT%H")
                self.prices[key + ":30"] = float(row["Hour 3B"])

        print(
            f"[SpotPriceProvider] Loaded {len(self.prices):,} hourly prices. "
            f"Range: €{min(self.prices.values()):.2f} – €{max(self.prices.values()):.2f}/MWh."
        )

    def get_price(self, dt: datetime) -> float | None:
        """Get price for a given UTC datetime (floored to hour)."""
        key = dt.strftime("%Y-%m-%dT%H")
        return self.prices.get(key)

    def get_prices_for_window(
        self, start: datetime, end: datetime
    ) -> list[tuple[datetime, float]]:
        """Return all hourly (timestamp, price) pairs within [start, end)."""
        slots: list[tuple[datetime, float]] = []
        current = start.replace(minute=0, second=0, microsecond=0)
        if current < start:
            current += timedelta(hours=1)

        while current < end:
            price = self.get_price(current)
            if price is not None:
                slots.append((current, price))
            current += timedelta(hours=1)

        return slots

    @property
    def coverage_start(self) -> str:
        return min(self.prices.keys())

    @property
    def coverage_end(self) -> str:
        return max(self.prices.keys())


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 — OPTIMAL CHARGE SCHEDULER
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class ScheduleResult:
    """Result of scheduling one charging session."""
    session: ChargingSession
    slots_available: int
    slots_needed: int
    naive_cost_eur: float         # cost if charging starts immediately
    optimal_cost_eur: float       # cost using cheapest available slots
    naive_avg_price: float        # avg EUR/MWh under naive strategy
    optimal_avg_price: float      # avg EUR/MWh under optimal strategy

    @property
    def savings_eur(self) -> float:
        return self.naive_cost_eur - self.optimal_cost_eur

    @property
    def savings_pct(self) -> float:
        if self.naive_cost_eur <= 0:
            return 0.0
        return self.savings_eur / self.naive_cost_eur * 100.0


class OptimalChargeScheduler:
    """
    For each session, determines the cheapest set of hourly slots to charge in.

    Strategy (hindsight-optimal / greedy):
        1. Enumerate all available hourly time slots within the plug-in window.
        2. Compute how many slots are needed: ceil(energy_needed / (rate × 1h)).
        3. Sort slots by ascending price.
        4. Pick the cheapest N slots.

    This is provably optimal when:
        - Charge rate is constant (no battery curve taper)
        - No switching/startup costs
        - No grid capacity constraints across vehicles

    For more complex scenarios, replace this with an LP solver.
    """

    def __init__(self, price_provider: SpotPriceProvider):
        self.price_provider = price_provider

    def schedule(self, session: ChargingSession) -> ScheduleResult | None:
        """
        Compute naive vs optimal cost for one session.
        Returns None if insufficient price data.
        """
        price_slots = self.price_provider.get_prices_for_window(
            session.plug_in_time, session.plug_out_time
        )

        if not price_slots:
            return None

        energy_per_slot = session.charge_rate_kw * 1.0  # kWh per 1-hour slot
        slots_needed = math.ceil(session.energy_needed_kwh / energy_per_slot)

        if slots_needed <= 0 or len(price_slots) < slots_needed:
            return None

        prices = [p for _, p in price_slots]

        # ── Naive strategy: charge in the first N slots ──
        naive_prices = prices[:slots_needed]
        naive_cost = sum(p * energy_per_slot / 1000.0 for p in naive_prices)
        naive_avg = float(np.mean(naive_prices))

        # ── Optimal strategy: charge in the cheapest N slots ──
        sorted_prices = sorted(prices)
        optimal_prices = sorted_prices[:slots_needed]
        optimal_cost = sum(p * energy_per_slot / 1000.0 for p in optimal_prices)
        optimal_avg = float(np.mean(optimal_prices))

        return ScheduleResult(
            session=session,
            slots_available=len(price_slots),
            slots_needed=slots_needed,
            naive_cost_eur=naive_cost,
            optimal_cost_eur=optimal_cost,
            naive_avg_price=naive_avg,
            optimal_avg_price=optimal_avg,
        )


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 — ANALYSIS ORCHESTRATOR
# ═══════════════════════════════════════════════════════════════════════════════

class ChargingAnalysis:
    """Full pipeline: load → extract → schedule → aggregate."""

    def __init__(
        self,
        ev_data_path: str | Path,
        spot_price_path: str | Path,
    ):
        self.loader = EVDataLoader()
        self.price_provider = SpotPriceProvider(spot_price_path)
        self.ev_data_path = ev_data_path

        self._events: pd.DataFrame | None = None
        self._sessions: list[ChargingSession] = []
        self._results: list[ScheduleResult] = []
        self._vehicles: dict[str, Vehicle] = {}

    def run(self) -> pd.DataFrame:
        """Execute the full pipeline and return a session-level summary DataFrame."""
        # Step 1: Load
        self._events = self.loader.load(self.ev_data_path)
        self._build_vehicle_registry()

        # Step 2: Extract sessions
        extractor = SessionExtractor(self._vehicles)
        self._sessions = extractor.extract(self._events)

        # Step 3 & 4: Schedule each session
        scheduler = OptimalChargeScheduler(self.price_provider)
        self._results = []
        skipped = 0

        for session in self._sessions:
            result = scheduler.schedule(session)
            if result is not None:
                self._results.append(result)
            else:
                skipped += 1

        print(
            f"[ChargingAnalysis] Scheduled {len(self._results):,} sessions. "
            f"Skipped {skipped:,} (insufficient price data or too short)."
        )

        return self._build_summary()

    def _build_vehicle_registry(self) -> None:
        """Build a Vehicle object for each unique vehicle_id."""
        df = self._events
        for vid, grp in df.groupby("vehicle_id"):
            first = grp.iloc[0]
            self._vehicles[vid] = Vehicle(
                vehicle_id=vid,
                brand=first["vehicle_information_brand"],
                model=first["vehicle_information_model"],
                year=(
                    int(first["vehicle_information_year"])
                    if pd.notna(first.get("vehicle_information_year"))
                    else None
                ),
                battery_capacity_kwh=(
                    first["vehicle_chargestate_batterycapacity"]
                    if pd.notna(first.get("vehicle_chargestate_batterycapacity"))
                    else None
                ),
            )

    def _build_summary(self) -> pd.DataFrame:
        """Create a session-level DataFrame with all results."""
        rows = []
        for r in self._results:
            s = r.session
            v = self._vehicles.get(s.vehicle_id)
            rows.append({
                "vehicle_id": s.vehicle_id,
                "brand": v.brand if v else "",
                "model": v.model if v else "",
                "plug_in_time": s.plug_in_time,
                "plug_out_time": s.plug_out_time,
                "hours_available": round(s.hours_available, 1),
                "soc_start_pct": s.soc_start_pct,
                "soc_target_pct": s.soc_target_pct,
                "battery_capacity_kwh": s.battery_capacity_kwh,
                "charge_rate_kw": s.charge_rate_kw,
                "energy_needed_kwh": round(s.energy_needed_kwh, 2),
                "hours_needed": round(s.hours_needed, 1),
                "flexibility_ratio": round(s.flexibility_ratio, 2),
                "slots_available": r.slots_available,
                "slots_needed": r.slots_needed,
                "naive_cost_eur": round(r.naive_cost_eur, 4),
                "optimal_cost_eur": round(r.optimal_cost_eur, 4),
                "savings_eur": round(r.savings_eur, 4),
                "savings_pct": round(r.savings_pct, 2),
                "naive_avg_price_eur_mwh": round(r.naive_avg_price, 2),
                "optimal_avg_price_eur_mwh": round(r.optimal_avg_price, 2),
            })

        return pd.DataFrame(rows)

    # ── Aggregate reports ──

    def fleet_summary(self, df: pd.DataFrame) -> dict:
        """Top-level fleet KPIs."""
        total_naive = df["naive_cost_eur"].sum()
        return {
            "total_vehicles": df["vehicle_id"].nunique(),
            "total_sessions": len(df),
            "total_energy_kwh": round(df["energy_needed_kwh"].sum(), 0),
            "total_naive_cost_eur": round(total_naive, 2),
            "total_optimal_cost_eur": round(df["optimal_cost_eur"].sum(), 2),
            "total_savings_eur": round(df["savings_eur"].sum(), 2),
            "total_savings_pct": round(
                df["savings_eur"].sum() / total_naive * 100, 1
            ) if total_naive > 0 else 0,
            "avg_savings_per_session_eur": round(df["savings_eur"].mean(), 2),
            "median_flexibility_ratio": round(df["flexibility_ratio"].median(), 2),
        }

    def by_model(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aggregate savings by vehicle model."""
        df = df.copy()
        df["display_model"] = df["brand"] + " " + df["model"]
        agg = (
            df.groupby("display_model")
            .agg(
                vehicles=("vehicle_id", "nunique"),
                sessions=("vehicle_id", "count"),
                energy_kwh=("energy_needed_kwh", "sum"),
                naive_cost_eur=("naive_cost_eur", "sum"),
                optimal_cost_eur=("optimal_cost_eur", "sum"),
                savings_eur=("savings_eur", "sum"),
            )
            .reset_index()
        )
        agg["savings_pct"] = np.where(
            agg["naive_cost_eur"] > 0,
            agg["savings_eur"] / agg["naive_cost_eur"] * 100,
            0,
        )
        agg = agg.sort_values("savings_eur", ascending=False).reset_index(drop=True)
        return agg.round(2)

    def by_duration_bucket(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aggregate savings by session duration bucket."""
        bins = [0, 2, 6, 12, 24, float("inf")]
        labels = ["< 2h", "2-6h", "6-12h", "12-24h", "> 24h"]
        df = df.copy()
        df["duration_bucket"] = pd.cut(
            df["hours_available"], bins=bins, labels=labels, right=False
        )
        agg = (
            df.groupby("duration_bucket", observed=True)
            .agg(
                sessions=("vehicle_id", "count"),
                naive_cost_eur=("naive_cost_eur", "sum"),
                optimal_cost_eur=("optimal_cost_eur", "sum"),
                savings_eur=("savings_eur", "sum"),
            )
            .reset_index()
        )
        agg["savings_pct"] = np.where(
            agg["naive_cost_eur"] > 0,
            agg["savings_eur"] / agg["naive_cost_eur"] * 100,
            0,
        )
        return agg.round(2)


# ═══════════════════════════════════════════════════════════════════════════════
# CLI ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description="EV Smart Charging — Hindsight Optimal Schedule Calculator"
    )
    parser.add_argument(
        "--ev-data", required=True,
        help="Path to EV telematics CSV (smart_charge_ev_2026.csv)",
    )
    parser.add_argument(
        "--spot-prices", required=True,
        help="Path to EPEX SPOT auction prices CSV",
    )
    parser.add_argument(
        "--output", default="charging_optimization_results.csv",
        help="Output CSV path (default: charging_optimization_results.csv)",
    )
    args = parser.parse_args()

    # ── Run the pipeline ──
    analysis = ChargingAnalysis(
        ev_data_path=args.ev_data,
        spot_price_path=args.spot_prices,
    )
    results_df = analysis.run()

    # ── Print reports ──
    print("\n" + "=" * 70)
    print("FLEET SUMMARY")
    print("=" * 70)
    fleet = analysis.fleet_summary(results_df)
    for k, v in fleet.items():
        label = k.replace("_", " ").title()
        print(f"  {label:.<40} {v}")

    print("\n" + "=" * 70)
    print("SAVINGS BY VEHICLE MODEL")
    print("=" * 70)
    model_df = analysis.by_model(results_df)
    print(model_df.to_string(index=False))

    print("\n" + "=" * 70)
    print("SAVINGS BY SESSION DURATION")
    print("=" * 70)
    dur_df = analysis.by_duration_bucket(results_df)
    print(dur_df.to_string(index=False))

    print("\n" + "=" * 70)
    print("TOP 10 SESSIONS BY SAVINGS")
    print("=" * 70)
    top = results_df.nlargest(10, "savings_eur")[
        [
            "brand", "model", "plug_in_time", "hours_available",
            "energy_needed_kwh", "naive_cost_eur", "optimal_cost_eur",
            "savings_eur", "savings_pct",
        ]
    ]
    print(top.to_string(index=False))

    # ── Export ──
    results_df.to_csv(args.output, index=False)
    print(f"\n✅ Session-level results exported to: {args.output}")


In [15]:
# ── Run directly in notebook (bypasses argparse) ──
EV_DATA_PATH = "Smart Charging Data/smart_charge_ev_2026.csv"
SPOT_PRICES_PATH = "auction_spot_prices/auction_spot_prices_switzerland_2026.csv"
OUTPUT_PATH = "charging_optimization_results.csv"

analysis = ChargingAnalysis(
    ev_data_path=EV_DATA_PATH,
    spot_price_path=SPOT_PRICES_PATH,
)
results_df = analysis.run()

print("\n" + "=" * 70)
print("FLEET SUMMARY")
print("=" * 70)
fleet = analysis.fleet_summary(results_df)
for k, v in fleet.items():
    label = k.replace("_", " ").title()
    print(f"  {label:.<40} {v}")

print("\n" + "=" * 70)
print("SAVINGS BY VEHICLE MODEL")
print("=" * 70)
model_df = analysis.by_model(results_df)
print(model_df.to_string(index=False))

print("\n" + "=" * 70)
print("SAVINGS BY SESSION DURATION")
print("=" * 70)
dur_df = analysis.by_duration_bucket(results_df)
print(dur_df.to_string(index=False))

print("\n" + "=" * 70)
print("TOP 10 SESSIONS BY SAVINGS")
print("=" * 70)
top = results_df.nlargest(10, "savings_eur")[[
    "brand", "model", "plug_in_time", "hours_available",
    "energy_needed_kwh", "naive_cost_eur", "optimal_cost_eur",
    "savings_eur", "savings_pct",
]]
print(top.to_string(index=False))

results_df.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Session-level results exported to: {OUTPUT_PATH}")

[SpotPriceProvider] Loaded 1,656 hourly prices. Range: €1.50 – €266.47/MWh.
[EVDataLoader] Reading Smart Charging Data/smart_charge_ev_2026.csv...
[EVDataLoader] Loaded 276,529 rows, 111 vehicles.
[SessionExtractor] Extracted 3,459 charging sessions.
[ChargingAnalysis] Scheduled 2,228 sessions. Skipped 1,231 (insufficient price data or too short).

FLEET SUMMARY
  Total Vehicles.......................... 111
  Total Sessions.......................... 2228
  Total Energy Kwh........................ 77686.0
  Total Naive Cost Eur.................... 11759.85
  Total Optimal Cost Eur.................. 10735.67
  Total Savings Eur....................... 1024.19
  Total Savings Pct....................... 8.7
  Avg Savings Per Session Eur............. 0.46
  Median Flexibility Ratio................ 2.42

SAVINGS BY VEHICLE MODEL
      display_model  vehicles  sessions  energy_kwh  naive_cost_eur  optimal_cost_eur  savings_eur  savings_pct
Volkswagen ID. Buzz        24       501    16192.80  